In [1]:
import pandas as pd

In [2]:
transactions_file_path = "transactions_data.csv"
transactions_data = pd.read_csv(transactions_file_path)

In [3]:
products_file_path = "products_data_cleaned.csv"
products_data = pd.read_csv(products_file_path)

In [4]:
if 'Unnamed: 0' in transactions_data.columns:
    transactions_data.drop(columns=['Unnamed: 0'], inplace=True)

In [5]:
transactions_data.dropna(subset=["Transaction_ID", "Company_ID", "Product_ID"], inplace=True)

In [6]:
average_quantity_per_product = transactions_data.groupby("Product_ID")["Quantity"].transform(lambda x: x.fillna(x.mean()))
transactions_data.loc[:, "Quantity"] = transactions_data["Quantity"].fillna(average_quantity_per_product).fillna(1)

In [7]:
product_price_map = products_data.set_index("Product_ID")["Product_Price"].to_dict()
transactions_data.loc[:, "Product_Price"] = transactions_data["Product_Price"].fillna(transactions_data["Product_ID"].map(product_price_map))

In [8]:
transactions_data.loc[:, "Total_Cost"] = transactions_data["Quantity"] * transactions_data["Product_Price"]

In [9]:
transactions_data.loc[:, "Quantity"] = transactions_data["Quantity"].round(2)
transactions_data.loc[:, "Product_Price"] = transactions_data["Product_Price"].round(2)
transactions_data.loc[:, "Total_Cost"] = transactions_data["Total_Cost"].round(2)

In [10]:
transactions_data.loc[:, "Transaction_Date"] = pd.to_datetime(transactions_data["Transaction_Date"], errors='coerce')

In [11]:
transactions_data.dropna(subset=["Transaction_Date"], inplace=True)

In [12]:
duplicates = transactions_data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

Number of duplicate rows: 0


In [14]:
print(transactions_data.isnull().sum())

Transaction_ID      0
Company_ID          0
Product_ID          0
Quantity            0
Transaction_Date    0
Product_Price       0
Total_Cost          0
dtype: int64


In [15]:
transactions_data.to_csv("transactions_data_cleaned.csv", index=False)